# 可选实验：线性回归的梯度下降

<figure>
    <center> <img src="./images/C1_W1_L4_S1_Lecture_GD.png"  style="width:800px;height:200px;" ></center>
</figure>

## 目标
在本实验中，你将：
- 使用梯度下降自动化优化 $w$ 和 $b$ 的过程。

## 工具
在本实验中，我们将使用：
- NumPy，一个流行的科学计算库
- Matplotlib，一个流行的数据可视化库
- 本地目录中 lab_utils.py 文件里的绘图例程

In [ ]:
import math, copy
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
from lab_utils_uni import plt_house_x, plt_contour_wgrad, plt_divergence, plt_gradients

<a name="toc_40291_2"></a>
# 问题描述

让我们使用与之前相同的两个数据点 - 一栋1000平方英尺的房子售价30万美元，一栋2000平方英尺的房子售价50万美元。

| 大小（1000平方英尺）     | 价格（1000美元） |
| ----------------| ------------------------ |
| 1               | 300                      |
| 2               | 500                      |


In [ ]:
# 加载我们的数据集
x_train = np.array([1.0, 2.0])   #特征
y_train = np.array([300.0, 500.0])   #目标值

<a name="toc_40291_2.0.1"></a>
### 计算代价
这是在上一个实验中开发的。我们在这里将再次需要它。

In [ ]:
#计算代价的函数
def compute_cost(x, y, w, b):
   
    m = x.shape[0] 
    cost = 0
    
    for i in range(m):
        f_wb = w * x[i] + b
        cost = cost + (f_wb - y[i])**2
    total_cost = 1 / (2 * m) * cost

    return total_cost

<a name="toc_40291_2.1"></a>
## 梯度下降总结
到目前为止，在本课程中，你已经开发了一个预测 $f_{w,b}(x^{(i)})$ 的线性模型：
$$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$
在线性回归中，你利用输入训练数据，通过最小化预测值 $f_{w,b}(x^{(i)})$ 与实际数据 $y^{(i)}$ 之间的误差度量来拟合参数 $w$,$b$。这个度量称为 $代价$ $J(w,b)$。在训练中，你在所有训练样本 $x^{(i)},y^{(i)}$ 上测量代价
$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2\tag{2}$$ 


在课程中，*梯度下降*被描述为：

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline
\;  w &= w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{3}  \; \newline 
 b &= b -  \alpha \frac{\partial J(w,b)}{\partial b}  \newline \rbrace
\end{align*}$$
其中，参数 $w$, $b$ 是同步更新的。
梯度定义为：
$$
\begin{align}
\frac{\partial J(w,b)}{\partial w}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})x^{(i)} \tag{4}\\
  \frac{\partial J(w,b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{5}\\
\end{align}
$$

这里的"*同步*"意味着你在更新任何参数之前先计算所有参数的偏导数。

<a name="toc_40291_2.2"></a>
## 实现梯度下降
你将为单个特征实现梯度下降算法。你需要三个函数。
- `compute_gradient` 实现上面的方程(4)和(5)
- `compute_cost` 实现上面的方程(2)（来自前面实验的代码）
- `gradient_descent`，使用 compute_gradient 和 compute_cost

约定：
- 包含偏导数的Python变量命名遵循以下模式，$\frac{\partial J(w,b)}{\partial b}$ 将命名为 `dj_db`。
- w.r.t 是 With Respect To（关于）的缩写，如 $J(wb)$ 关于 $b$ 的偏导数。


<a name="toc_40291_2.3"></a>
### compute_gradient
<a name='ex-01'></a>
`compute_gradient` 实现上面的方程(4)和(5)，并返回 $\frac{\partial J(w,b)}{\partial w}$,$\frac{\partial J(w,b)}{\partial b}$。内嵌的注释描述了操作。

In [ ]:
def compute_gradient(x, y, w, b): 
    """
    计算线性回归的梯度
    参数：
      x (ndarray (m,)): 数据，m个样本
      y (ndarray (m,)): 目标值
      w,b (标量)      : 模型参数
    返回：
      dj_dw (标量): 代价关于参数w的梯度
      dj_db (标量): 代价关于参数b的梯度
     """
    
    # 训练样本数量
    m = x.shape[0]    
    dj_dw = 0
    dj_db = 0
    
    for i in range(m):  
        f_wb = w * x[i] + b 
        dj_dw_i = (f_wb - y[i]) * x[i] 
        dj_db_i = f_wb - y[i] 
        dj_db += dj_db_i
        dj_dw += dj_dw_i 
    dj_dw = dj_dw / m 
    dj_db = dj_db / m 
        
    return dj_dw, dj_db

<br/>

<img align="left" src="./images/C1_W1_Lab03_lecture_slopes.PNG"   style="width:340px;" > 课程中描述了梯度下降如何利用代价关于某参数在某点的偏导数来更新该参数。
让我们使用 `compute_gradient` 函数来找到并绘制代价函数关于其中一个参数 $w_0$ 的一些偏导数。


In [ ]:
plt_gradients(x_train,y_train, compute_cost, compute_gradient)
plt.show()

上面，左图显示了 $\frac{\partial J(w,b)}{\partial w}$，即代价曲线在三个点处相对于 $w$ 的斜率。在图的右侧，导数为正，而在左侧为负。由于"碗形"的特性，导数将始终引导梯度下降朝向梯度为零的底部。
 
左图固定了 $b=100$。梯度下降将同时利用 $\frac{\partial J(w,b)}{\partial w}$ 和 $\frac{\partial J(w,b)}{\partial b}$ 来更新参数。右边的"箭头图"提供了查看两个参数梯度的方法。箭头的大小反映了该点梯度的大小。箭头的方向和斜率反映了该点 $\frac{\partial J(w,b)}{\partial w}$ 和 $\frac{\partial J(w,b)}{\partial b}$ 的比值。
注意梯度指向*远离*最小值的方向。回顾上面的方程(3)。缩放后的梯度从 $w$ 或 $b$ 的当前值中*减去*。这会将参数移动到减少代价的方向。

<a name="toc_40291_2.5"></a>
### 梯度下降
现在梯度可以计算了，上面方程(3)中描述的梯度下降可以在下面的 `gradient_descent` 中实现。实现的细节在注释中描述。下面，你将使用这个函数在训练数据上找到 $w$ 和 $b$ 的最优值。

In [ ]:
def gradient_descent(x, y, w_in, b_in, alpha, num_iters, cost_function, gradient_function): 
    """
    执行梯度下降来拟合w,b。通过以学习率alpha执行
    num_iters次梯度步骤来更新w,b
    
    参数：
      x (ndarray (m,))  : 数据，m个样本
      y (ndarray (m,))  : 目标值
      w_in,b_in (标量)  : 模型参数的初始值
      alpha (浮点数)    : 学习率
      num_iters (整数)  : 运行梯度下降的迭代次数
      cost_function     : 产生代价的函数
      gradient_function : 产生梯度的函数
      
    返回：
      w (标量): 运行梯度下降后参数的更新值
      b (标量): 运行梯度下降后参数的更新值
      J_history (列表): 代价值的历史记录
      p_history (列表): 参数 [w,b] 的历史记录
      """
    
    # 一个数组，用于存储每次迭代的代价J和w值，主要用于后续绘图
    J_history = []
    p_history = []
    b = b_in
    w = w_in
    
    for i in range(num_iters):
        # 计算梯度并使用gradient_function更新参数
        dj_dw, dj_db = gradient_function(x, y, w , b)     

        # 使用上面的方程(3)更新参数
        b = b - alpha * dj_db                            
        w = w - alpha * dj_dw                            

        # 每次迭代保存代价J
        if i<100000:      # 防止资源耗尽
            J_history.append( cost_function(x, y, w , b))
            p_history.append([w,b])
        # 每隔一定间隔打印代价，共10次，如果迭代次数<10则打印所有
        if i% math.ceil(num_iters/10) == 0:
            print(f"Iteration {i:4}: Cost {J_history[-1]:0.2e} ",
                  f"dj_dw: {dj_dw: 0.3e}, dj_db: {dj_db: 0.3e}  ",
                  f"w: {w: 0.3e}, b:{b: 0.5e}")
 
    return w, b, J_history, p_history #return w and J,w history for graphing

In [ ]:
# 初始化参数
w_init = 0
b_init = 0
# 设置梯度下降参数
iterations = 10000
tmp_alpha = 1.0e-2
# 运行梯度下降
w_final, b_final, J_hist, p_hist = gradient_descent(x_train ,y_train, w_init, b_init, tmp_alpha, 
                                                    iterations, compute_cost, compute_gradient)
print(f"(w,b) found by gradient descent: ({w_final:8.4f},{b_final:8.4f})")

<img align="left" src="./images/C1_W1_Lab03_lecture_learningrate.PNG"  style="width:340px; padding: 15px; " > 
花点时间注意上面打印的梯度下降过程的一些特征。

- 代价开始时很大，然后如课程幻灯片中所述迅速下降。
- 偏导数 `dj_dw` 和 `dj_db` 也变小了，起初很快，然后更慢。如课程图中所示，当过程接近"碗底"时，由于该点导数值较小，进展变慢。
- 尽管学习率alpha保持不变，进展仍然变慢

### 代价与梯度下降迭代次数
代价与迭代次数的关系图是衡量梯度下降进展的有用指标。在成功的运行中，代价应该总是减少的。代价的变化最初非常快，将初始下降和最终下降绘制在不同的比例上是有用的。在下面的图中，请注意坐标轴上代价的比例和迭代步长。

In [ ]:
# 绘制代价与迭代次数的关系图
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12,4))
ax1.plot(J_hist[:100])
ax2.plot(1000 + np.arange(len(J_hist[1000:])), J_hist[1000:])
ax1.set_title("Cost vs. iteration(start)");  ax2.set_title("Cost vs. iteration (end)")
ax1.set_ylabel('Cost')            ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')  ;  ax2.set_xlabel('iteration step') 
plt.show()

### 预测
现在你已经找到了参数 $w$ 和 $b$ 的最优值，你可以使用模型根据我们学到的参数来预测房屋价值。正如预期的那样，预测值与相同房屋的训练值几乎相同。此外，不在训练集中的预测值也与期望值一致。

In [ ]:
print(f"1000 sqft house prediction {w_final*1.0 + b_final:0.1f} Thousand dollars")
print(f"1200 sqft house prediction {w_final*1.2 + b_final:0.1f} Thousand dollars")
print(f"2000 sqft house prediction {w_final*2.0 + b_final:0.1f} Thousand dollars")

<a name="toc_40291_2.6"></a>
## 绘图
你可以通过在代价(w,b)的等高线图上绘制代价随迭代次数的变化来显示梯度下降执行过程中的进展。

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 6))
plt_contour_wgrad(x_train, y_train, p_hist, ax)

上面，等高线图显示了在一定范围的 $w$ 和 $b$ 上的 $cost(w,b)$。代价水平由圆环表示。叠加的红色箭头是梯度下降的路径。以下是一些值得注意的要点：
- 路径朝着目标稳步（单调）前进。
- 初始步骤比接近目标时的步骤大得多。

**放大**，我们可以看到梯度下降的最后步骤。注意随着梯度接近零，步骤之间的距离缩小了。

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 4))
plt_contour_wgrad(x_train, y_train, p_hist, ax, w_range=[180, 220, 0.5], b_range=[80, 120, 0.5],
            contours=[1,5,10,20],resolution=0.5)

<a name="toc_40291_2.7.1"></a>
### 增大学习率

<figure>
 <img align="left", src="./images/C1_W1_Lab03_alpha_too_big.PNG"   style="width:340px;height:240px;" >
</figure>
在课程中，有关于方程(3)中学习率 $\alpha$ 合适值的讨论。$\alpha$ 越大，梯度下降收敛到解的速度就越快。但如果太大，梯度下降将会发散。上面你有一个很好收敛的解的例子。

让我们尝试增大 $\alpha$ 的值，看看会发生什么：

In [ ]:
# 初始化参数
w_init = 0
b_init = 0
# 将alpha设置为较大的值
iterations = 10
tmp_alpha = 8.0e-1
# 运行梯度下降
w_final, b_final, J_hist, p_hist = gradient_descent(x_train ,y_train, w_init, b_init, tmp_alpha, 
                                                    iterations, compute_cost, compute_gradient)

上面，$w$ 和 $b$ 在正负之间来回跳动，且绝对值随每次迭代增大。此外，每次迭代 $\frac{\partial J(w,b)}{\partial w}$ 改变符号，代价在增加而不是减少。这是*学习率太大*且解正在发散的明显迹象。
让我们用图来可视化这个过程。

In [ ]:
plt_divergence(p_hist, J_hist,x_train, y_train)
plt.show()

上面，左图显示了梯度下降前几步中 $w$ 的变化过程。$w$ 在正负之间振荡，代价快速增长。梯度下降同时作用于 $w$ 和 $b$，所以需要右边的3D图来获得完整的图景。


## 恭喜！
在本实验中，你：
- 深入了解了单变量梯度下降的细节。
- 开发了计算梯度的例程
- 可视化了梯度是什么
- 完成了梯度下降例程
- 利用梯度下降找到了参数
- 检验了学习率大小的影响